## Work with GPU

In [ ]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
import optuna

In [ ]:
torch.cuda.is_available()

True

In [ ]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.11.0+cu128
12.8
True
NVIDIA GeForce RTX 4050 Laptop GPU


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device being used:',device)

Device being used: cuda


In [ ]:
device

device(type='cuda')

In [ ]:
df = pd.read_csv("fashion-mnist_train.csv")
df.shape

(60000, 785)

In [ ]:
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
features = df.iloc[:,1:]
targets = df.iloc[:,0]
targets

0        2
1        9
2        6
3        0
4        3
        ..
59995    9
59996    1
59997    8
59998    8
59999    7
Name: label, Length: 60000, dtype: int64

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(features, targets, test_size=0.2, random_state= 42)
x_train /= 255.0
x_test /= 255.0

In [ ]:
x_train.shape

(48000, 784)

In [ ]:
x_test.shape

(12000, 784)

In [ ]:
device

device(type='cuda')

In [ ]:
# to create the dataset
class CustomDataset(Dataset):
    def __init__(self, features, targets):
        self.features = torch.tensor(features.to_numpy()).to(torch.float32)
        self.targets = torch.tensor(targets.to_numpy()).to(torch.long)
    
    def __len__(self):
        return self.features.shape[0]
    
    def __getitem__(self, index):
        return (self.features[index], self.targets[index])

In [ ]:
# dataset collection stored here
train_dataset = CustomDataset(x_train, y_train)
test_dataset = CustomDataset(x_test, y_test)

In [ ]:
train_dataset.__len__()

48000

In [ ]:
test_dataset.__len__()

12000

In [ ]:
train_dataset.__getitem__(0)

(tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.2275,
         0.5333, 0.0000, 0.0

In [18]:
train_dataset_loader = DataLoader(train_dataset, batch_size= 32, shuffle= True, pin_memory=True)
test_dataset_loader = DataLoader(test_dataset, batch_size= 32, shuffle=False, pin_memory=True)

In [19]:
for feature, target in train_dataset_loader:
    print(feature)
    print(target)
    break

tensor([[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.0000, 0.0000, 0.0000,  ..., 0.0941, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]])
tensor([3, 5, 1, 6, 5, 1, 6, 0, 9, 2, 1, 5, 1, 4, 9, 7, 4, 2, 3, 9, 0, 3, 8, 6,
        2, 1, 1, 3, 8, 2, 4, 1])


In [ ]:
class ImageClassifier(nn.Module):

    def __init__(self, number_of_features):
        super().__init__()

        # Image classifier model
        self.model = nn.Sequential(
            nn.Linear(number_of_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p = 0.4),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p = 0.4),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(p = 0.4),
            nn.Linear(32, 10)
        )

        # optimizer
        self.optim = optim.SGD(self.model.parameters(), lr = 0.1, weight_decay= 1e-4)
    
    def forward(self, features):
        self.y_pred = self.model(features)
        return self.y_pred
    
    def loss_function(self, y_real):
        loss_fn = nn.CrossEntropyLoss()
        self.loss = loss_fn(self.y_pred, y_real)
        return self.loss.item()
    
    def optimization(self):
        self.optim.zero_grad()
        self.loss.backward()
        self.optim.step()


In [ ]:
x_train.shape[1]

In [ ]:
# model is defined but the model uses the "cpu"
model = ImageClassifier(x_train.shape[1])

In [ ]:
for i in model.named_parameters():
    print(i)

In [ ]:
device

In [ ]:
# making model use gpu instead of cpu
model = model.to(device)

In [ ]:
model

In [ ]:
epochs = 30
total_loss = 0
for epoch in range(epochs):
    epoch += 1
    total_loss = 0
    for features_batch, targets_batch in train_dataset_loader:
        features_batch = features_batch.to(device)
        targets_batch = targets_batch.to(device)
        y_pred = model(features_batch)
        loss = model.loss_function(targets_batch)
        model.optimization()
        total_loss = total_loss + loss

    print("epoch:", epoch, "\tloss:", total_loss/len(train_dataset_loader))

In [ ]:
# model ready for evaluation
model.eval()

### Accuracy:
We need to find the accurcy now. The accuracy of the data is given by the formula:

Accuracy = (number of correct predictions) / (total number of the predictions)

In [ ]:
len(test_dataset_loader)*32

In [ ]:
correct = 0
data_records = 0
with torch.no_grad():
    for features_batch, targets_batch in test_dataset_loader:
        features_batch = features_batch.to(device)
        targets_batch = targets_batch.to(device)
        y_pred = model(features_batch)
        correct = correct + (targets_batch == torch.max(y_pred, dim = 1)[1]).sum().item()
        data_records = data_records + features_batch.shape[0]
    print("total data_records:", data_records, "correct records:", correct)
    accuracy = correct / data_records
    print(accuracy)

In [ ]:
correct = 0
data_records = 0
with torch.no_grad():
    for features_batch, targets_batch in train_dataset_loader:
        features_batch = features_batch.to(device)
        targets_batch = targets_batch.to(device)
        y_pred = model(features_batch)
        correct = correct + (targets_batch == torch.max(y_pred, dim = 1)[1]).sum().item()
        data_records = data_records + features_batch.shape[0]
    print("total data_records:", data_records, "correct records:", correct)
    accuracy = correct / data_records
    print(accuracy)

## Hyperparameter Tuning using Optuna:
Number of layers

Number of hidden neurons

numer of epochs

optimizer

learning rate

batch size

dropout

weight decay

In [29]:
# defining the class for ImgaeClassifier but in a different way:
# for optuna
class ImageClassifier(nn.Module):

    def __init__(self, no_of_features, no_of_outputs, no_of_hidden_layers, neurons_per_layer, dropout_rate):
        super().__init__()

        layers = []

        for i in range(no_of_hidden_layers):
            layers.append(nn.Linear(no_of_features, neurons_per_layer))
            layers.append(nn.BatchNorm1d(neurons_per_layer))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            no_of_features = neurons_per_layer
        layers.append(nn.Linear(neurons_per_layer, no_of_outputs))

        # model
        self.model = nn.Sequential(*layers)
    
    def forward(self, features):
        return self.model(features)

### defining the objective function

search space

model init

parameter init

training loop

evaluation loop

In [42]:
# defining the objective function

def objective(trial):
    # next hyperparameter value from the search space
    number_of_hidden_layers = trial.suggest_int("num_hidden_layers", 1, 5)
    number_of_neurons = trial.suggest_int("num_of_neurons", 8, 128, step=8)
    epochs = trial.suggest_int("epochs", 20, 50, step = 10)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-1, log = True)
    dropout_rate = trial.suggest_float("droput_rate", 0.1, 0.5, step = 0.1)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128])
    optimizer_name = trial.suggest_categorical("optimizer", ['Adam', 'SGD', 'RMSprop'])
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log = True)

    # dataset batch selection
    train_dataset_loader = DataLoader(train_dataset, batch_size, shuffle= True, pin_memory=True)
    test_dataset_loader = DataLoader(test_dataset, batch_size, shuffle=False, pin_memory=True) 

    # model init
    input_dim = 784
    output_dim = 10

    model = ImageClassifier(input_dim, output_dim, number_of_hidden_layers, number_of_neurons, dropout_rate) #cpu based model
    model.to(device) #gpu based model

    # loss function
    loss_fn = nn.CrossEntropyLoss()
    # optimization + l2 regularization
    if optimizer_name == "SGD":
        optim = torch.optim.SGD(model.parameters(), lr= learning_rate, weight_decay= weight_decay)
    elif optimizer_name == "Adam":
        optim = torch.optim.Adam(model.parameters(), lr= learning_rate, weight_decay= weight_decay)
    else:
        optim = torch.optim.RMSprop(model.parameters(), lr= learning_rate, weight_decay=weight_decay)

    # training loop
    for epoch in range(epochs):
        epoch += 1
        for features_batch, targets_batch in train_dataset_loader:
            # change the cpu data to gpu
            features_batch = features_batch.to(device)
            targets_batch = targets_batch.to(device)
            # forward pass
            y_pred = model(features_batch)
            # find the loss
            loss = loss_fn(y_pred, targets_batch)
            # remove all the gradients
            optim.zero_grad()
            # gradient
            loss.backward()
            # update the weight and bias
            optim.step()

    # evaluation loop
    model.eval()
    correct = 0
    data_records = 0
    with torch.no_grad():
        for features_batch, targets_batch in test_dataset_loader:
            features_batch = features_batch.to(device)
            targets_batch = targets_batch.to(device)
            y_pred = model(features_batch)
            correct = correct + (targets_batch == torch.max(y_pred, dim = 1)[1]).sum().item()
            data_records = data_records + features_batch.shape[0]
        accuracy = correct / data_records

    # return accruacy
    return accuracy

In [ ]:
study = optuna.create_study(direction="maximize")

In [ ]:
study.optimize(objective, n_trials=10)

In [ ]:
study.best_params()